# Notebook for comparing all implementations vs sklearn

In [1]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor as SklearnRF

#import custom implementations
from src_nico.customRegressionTreeForest import RandomForestNico as RFNico
from src_gregor.random_forest import RandomForestRegressor as RFGregor
from src_lenard.randomforest import RandomForestRegressor as RFLenard

#for now comparing just the random forest implementations, since they are already using the trees
#from src_nico.customRegressionTreeForest import DecisionTreeRegressor as DTNico
#from src_gregor.regression_tree import DecisionTreeRegressor  as  DTGregor
#from src_lenard.decisiontree import DecisionTreeRegressor as DTLenard

Since Nico uses a different input then the other methods, we created a quick wrapper to standardize things.

In [2]:
class NicoForestWrapper:
    """
    Wrapper to make Nico's implementation compatible with standard (X, y) calls
    used by sklearn and the other custom implementations.
    """
    def __init__(self, n_estimators=10, min_samples_split=2, max_depth=None, random_state=None):
        self.model = RFNico(random_state=random_state)
        self.n_estimators = n_estimators
        self.min_instances = min_samples_split
        self.max_depth = max_depth

    def fit(self, X, y):
        #Nico's model expects a single DataFrame containing both features and target
        data = pd.DataFrame(X).copy()
        #rename columns to strings to avoid potential issues
        data.columns = [f"feat_{i}" for i in range(data.shape[1])]
        target_name = "target_variable"
        data[target_name] = y
        
        self.model.fit(
            data=data,
            target_name=target_name,
            nr_of_trees=self.n_estimators,
            min_instances=self.min_instances,
            max_depth=self.max_depth
        )
        self.feature_names = data.columns.drop(target_name)
        return self

    def predict(self, X):
        #Nico's predict expects a DataFrame
        df = pd.DataFrame(X)
        df.columns = [f"feat_{i}" for i in range(df.shape[1])]
        return self.model.predict(df).values

### Evaluation function

In [3]:
def evaluate_model(model_name, model_instance, X, y, n_splits=5):
    """
    Performs K-Fold Cross Validation and logs performance metrics + timing.
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    rmse_scores = []
    r2_scores = []
    train_times = []
    pred_times = []
    
    print(f"--- Evaluating {model_name} ({n_splits}-Fold CV) ---")
    
    fold = 1
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        
        # 1. Measure Training Time
        start_train = time.time()
        model_instance.fit(X_train, y_train)
        end_train = time.time()
        train_duration = end_train - start_train
        train_times.append(train_duration)
        
        # 2. Measure Prediction Time
        start_pred = time.time()
        y_pred = model_instance.predict(X_test)
        end_pred = time.time()
        pred_duration = end_pred - start_pred
        pred_times.append(pred_duration)
        
        # 3. Calculate Metrics
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)
        
        rmse_scores.append(rmse)
        r2_scores.append(r2)
        
        print(f"  Fold {fold}: RMSE={rmse:.4f}, R2={r2:.4f}, Train Time={train_duration:.4f}s")
        fold += 1

    # Aggregated Results
    results = {
        "Model": model_name,
        "Avg RMSE": np.mean(rmse_scores),
        "Avg R2": np.mean(r2_scores),
        "Avg Train Time (s)": np.mean(train_times),
        "Avg Predict Time (s)": np.mean(pred_times)
    }
    return results

### Execution

In [4]:
#DATA LOADING (could be improved so its more general)
corn = pd.read_csv("data/corn_data_preprocessed.csv")
y = corn['Yield'].values
X = corn.drop('Yield', axis=1).values

#ensure y is 1D array
y = y.flatten()

#DEFINE HYPERPARAMETERS
N_TREES = 10
MAX_DEPTH = 5
MIN_SAMPLES = 2
SEED = 42

#INITIALIZE MODELS
models = [
    (
        "Sklearn RF",
        SklearnRF(n_estimators=N_TREES, max_depth=MAX_DEPTH, min_samples_split=MIN_SAMPLES, random_state=SEED, n_jobs=-1)
    ),
    (
        "Nico RF",
        NicoForestWrapper(n_estimators=N_TREES, max_depth=MAX_DEPTH, min_samples_split=MIN_SAMPLES, random_state=SEED)
    ),
    (
        "Gregor RF",
        RFGregor(n_estimators=N_TREES, max_depth=MAX_DEPTH, min_samples_split=MIN_SAMPLES, random_state=SEED)
    ),
    (
        "Lenard RF",
        RFLenard(n_estimators=N_TREES, max_depth=MAX_DEPTH, min_samples_split=MIN_SAMPLES)
    )
]

#RUN COMPARISON
all_results = []

for name, model in models:
    try:
        res = evaluate_model(name, model, X, y)
        all_results.append(res)
    except Exception as e:
        print(f"!! Failed to evaluate {name}: {e}")
        #if a model fails (e.g. recursion depth or type error), we log it empty
        all_results.append({
            "Model": name, "Avg RMSE": np.nan, "Avg R2": np.nan, 
            "Avg Train Time (s)": np.nan, "Avg Predict Time (s)": np.nan
        })

#DISPLAY FINAL TABLE
results_df = pd.DataFrame(all_results)
print("\n\n================ FINAL COMPARISON RESULTS ================")
print(results_df.round(4).to_string(index=False))
print("==========================================================")

--- Evaluating Sklearn RF (5-Fold CV) ---
  Fold 1: RMSE=69.1236, R2=0.7201, Train Time=0.0215s
  Fold 2: RMSE=53.8182, R2=0.8644, Train Time=0.0148s
  Fold 3: RMSE=48.9544, R2=0.8706, Train Time=0.0153s
  Fold 4: RMSE=52.2548, R2=0.8411, Train Time=0.0150s
  Fold 5: RMSE=42.0275, R2=0.9093, Train Time=0.0152s
--- Evaluating Nico RF (5-Fold CV) ---
!! Failed to evaluate Nico RF: RandomForestNico.fit() got an unexpected keyword argument 'data'
--- Evaluating Gregor RF (5-Fold CV) ---
  Fold 1: RMSE=70.0703, R2=0.7124, Train Time=1.4861s
  Fold 2: RMSE=54.4348, R2=0.8612, Train Time=0.0249s
  Fold 3: RMSE=51.4732, R2=0.8569, Train Time=0.0274s
  Fold 4: RMSE=51.7845, R2=0.8440, Train Time=0.0263s
  Fold 5: RMSE=38.9605, R2=0.9221, Train Time=0.0261s
--- Evaluating Lenard RF (5-Fold CV) ---
  Fold 1: RMSE=68.1589, R2=0.7278, Train Time=0.1860s
  Fold 2: RMSE=56.4460, R2=0.8508, Train Time=0.1667s
  Fold 3: RMSE=46.5786, R2=0.8828, Train Time=0.1630s
  Fold 4: RMSE=54.1746, R2=0.8293, Trai

- There are errors occuring in Lenard's calculations - should be looked into and fixed
- When run for the first time Nico's first fold always takes ~10 seconds. This is because `joblib` needs to create all the individual processes to properly parallelize the training process. The subsequent folds are much faster. If we run the evaluation again, the notebook keeps these processes saved, so all the folds are done equally fast.

### Test area for GridsearchCV adaption

In [5]:
from src_gregor.regression_tree import DecisionTreeRegressor as TreeGregor
from src_gregor.random_forest import RandomForestRegressor as RFGregor

import pandas as pd
import matplotlib.pyplot as plt

# for testing and comparison
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import make_regression, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

# for, well typing
from typing import Dict, Tuple
from sklearn.model_selection import  GridSearchCV
from sklearn.pipeline import Pipeline

def metrics_dict(y_true, y_pred) -> Dict[str, float]:
    return {
        "MSE": float(mean_squared_error(y_true, y_pred)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MAE_pct": float(mean_absolute_percentage_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred))
    }

def split_df(df: pd.DataFrame, 
             target_name: str, 
             test_size: float = 0.2, 
             random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    X = df.drop(columns=[target_name])
    y = df[target_name]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

    return X_train, X_test, y_train, y_test

def evaluate_run(name, task_type, y_true, y_pred):
    return dict(
            id=id,
            name=name,
            task_type = task_type,
            r2=r2_score(y_true, y_pred), 
            mse = mean_squared_error(y_true, y_pred), 
            mae = mean_absolute_error(y_true, y_pred), 
            mape = mean_absolute_percentage_error(y_true, y_pred)
        )

data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
# sklearn's fetch returns target in data.target; ensure a column name
if "target" not in df.columns:
    df["target"] = data.target
    target_col = "target"
else:
    target_col = df.columns[-1]

df

X_train, X_test, y_train, y_test = split_df(df, target_name = "target")

param_grid = {"classifier__max_depth": [3,5,7]}

pipe = Pipeline([
    ("classifier", RFGregor(n_estimators=5))
])

# gridwork whoop
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="r2",
    refit=True,
    return_train_score=False
)

grid.fit(X_train, y_train)
best_pipe = grid.best_estimator_
best_params = grid.best_params_

y_pred = best_pipe.predict(X_test)

metrics_dict(y_test, y_pred)




{'MSE': 3.707310132796373e-05,
 'MAE': 0.004272859628417673,
 'MAE_pct': 0.0037058779355622785,
 'R2': 0.9999717087550959}

In [8]:
print(best_pipe)
print(best_pipe.named_steps["classifier"].trees_)


Pipeline(steps=[('classifier',
                 RandomForestRegressor(max_depth=7, n_estimators=5))])
[DecisionTreeRegressor(max_depth=7), DecisionTreeRegressor(max_depth=7), DecisionTreeRegressor(max_depth=7), DecisionTreeRegressor(max_depth=7), DecisionTreeRegressor(max_depth=7)]
